# Parallel Jacobian calculations using MPI

This notebook shows how to benchmark pyGSTi's MPI-accelerated Jacobian calculations for a "full TP" parameterized gate set.

It's a three-step process.

1. Generate an experiment design and write it to disk. You can skip this step if you already have an experiment design on hand.
2. Build a large f-string representing the contents of a Python file; write that file to disk. If you skipped step 1 then you'll 
   need to provide your own value for ``edesign_parent_dir`` when we build the f-string.
3. Show the results of launching the new Python file with ``mpiexec``, varying the number of workers.


In [ ]:
####################################
# Step 1.
####################################

from pygsti.modelpacks import smq1Q_XYI as mp
from pygsti.io import write_empty_protocol_data
import os

edesign_parent_dir = os.getcwd() +  '/mpi_files'
exp_design = mp.create_gst_experiment_design(max_max_length=256)
write_empty_protocol_data(edesign_parent_dir, exp_design, clobber_ok=True)


In [ ]:
####################################
# Step 2.
####################################

mpi_script_contents = f"""
import time
import pygsti
from pygsti.baseobjs.resourceallocation import ResourceAllocation
from pygsti.tools.sharedmemtools import shared_mem_is_enabled
import numpy as np

from mpi4py import MPI

protocol_dir  = '{edesign_parent_dir}'
#
#  ^ That directory must include the following as subdirectories and files.
#    (It can also include other files, like this Python file.)
#
#    ├── data
#    │   ├── dataset.txt
#    │   └── meta.json
#    └── edesign
#        ├── all_circuits_needing_data.json
#           ...
#        ├── germs.txt
#        ├── meas_fiducials.txt
#        ├── meta.json
#        ├── prep_fiducials.txt
#        ├── processor_spec.json
#        └── subdirs.json
#
edesign = pygsti.io.read_edesign_from_dir( protocol_dir )
m = pygsti.models.create_explicit_model(edesign.processor_spec)
m.convert_members_inplace('full TP')
m.sim = 'map'
circuits = edesign.all_circuits_needing_data

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
print(f"Rank {{rank}} started")

resalloc = ResourceAllocation(comm, distribute_method='default')
assert shared_mem_is_enabled()
resalloc.build_hostcomms()
# ^ see pygsti/protocols/gst.py::(1371--1374)

layout = m.sim.create_layout(
    circuits, array_types=('ep',),
    resource_alloc=resalloc, verbosity=0
)
# ^ see pygsti/algorithms/core.py::895

dprobs_array = layout.allocate_local_array('ep', 'd')
# ^ Needed at pygsti/objectivefns/objectivefns.py::(4463--4467)
#    ^ Allocated at pygsti/objectivefns/objectivefns.py::4290

comm.Barrier()
if resalloc.is_host_leader:
    print('Total circuits : ' + str(len(circuits)))
    start = time.time()

m.sim.bulk_fill_dprobs(dprobs_array, layout)

comm.Barrier()
if resalloc.is_host_leader:
    end = time.time()
    full_dprobs = np.vstack([arr for arr in dprobs_array.host_array.values()])
    print(f'||Jac||_Fro^2  : {{np.linalg.norm(full_dprobs)**2}}')
    print(f'WALLCLOCK_TIME : {{end - start}}')
    full_dprobs = None

resalloc.host_comm_barrier()
layout.free_local_array(dprobs_array)
dprobs_array  = None
dprobs_arrays = []

"""
mpiexec_script = edesign_parent_dir + "/mpi_dprobs_script.py"
with open(mpiexec_script,"w") as f:
    f.write(mpi_script_contents)

## Step 3

We run the script with varying numbers of processors using `mpiexec`. 
You may need to replace it with ``mpirun`` depending on your system.

The script prints out the squared Frobenius norm of the Jacobian.
This is a sanity check that the whole Jacobian got computed, rather than only a portion of it.
This should be the same regardless of the number of MPI processors.

The script prints a value for WALLCLOCK_TIME. That value represents our expected
wallclock time that pyGSTi's GST model fitting optimizer will spend on a single
Jacobian calculation. During the optimization process we usually compute a few
hundred Jacobians.


In [ ]:
! which mpiexec; mpiexec -n 1 python {mpiexec_script}

In [ ]:
! which mpiexec; mpiexec -n 2 python {mpiexec_script}

In [ ]:
! which mpiexec; mpiexec -n 4 python {mpiexec_script}